# 제주 300m 그리드 파이프라인 통합 실행

ZIP에서 기본 월별 CSV 12개만 선택하여 긴 형태로 변환하고, Parquet과 프로파일 메타데이터로 저장합니다.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
import re

import pandas as pd

from src.tour_data_etl import TourDataETL
from src.data_profiler import DataProfiler

input_path = Path("data") / "제주특별자치도_시간대별_관광객_300그리드단위_유동인구.zip"
parquet_output_path = Path("outputs") / "jeju_grid_300m.parquet"
metadata_output_path = Path("outputs") / "jeju_grid_300m_metadata.json"

print("입력 파일:", input_path)
print("입력 파일 존재 여부:", input_path.exists())

입력 파일: data\제주특별자치도_시간대별_관광객_300그리드단위_유동인구.zip
입력 파일 존재 여부: True


In [ ]:
etl = TourDataETL()
profiler = DataProfiler()

# 날씨·평일·휴일 조건이 붙지 않은 기본 월별 CSV만 선택
base_file_pattern = re.compile(
    r"_비율포함_(\d{8})\.csv$"
)

with ZipFile(input_path) as zip_file:
    base_csv_files = sorted(
        file_name
        for file_name in zip_file.namelist() # 내부의 모든 파일명을 하나씩 확인
        if base_file_pattern.search(file_name) # 파일명이 정규표현식 패턴과 일치하는지 확인 -> 선택
    )

print("선택된 기본 월별 CSV 수:", len(base_csv_files))

for file_name in base_csv_files:
    print(Path(file_name).name)

assert len(base_csv_files) == 12

선택된 기본 월별 CSV 수: 12
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140131.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140228.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140331.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140430.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140531.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140630.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140731.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140831.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140930.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141031.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141130.csv
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141231.csv


In [ ]:
# 12개월 데이터를 시간대별 행 구조로 변환
monthly_frames = [] # 월별 데이터프레임을 담을 리스트

# 월별 CSV 파일을 하나씩 읽어와서 시간대별 행 구조로 변환
with ZipFile(input_path) as zip_file:
    for file_name in base_csv_files:
        # ZIP 파일 내부의 CSV 파일을 열고 pandas로 읽어오기
        with zip_file.open(file_name) as csv_file:
            monthly_df = pd.read_csv(
                csv_file,
                encoding="cp949",
            )

        # 월별로 다른 시간 구분자와 격자 컬럼명을 통일
        monthly_df.columns = [
            column.replace("-", "~")
            for column in monthly_df.columns
        ]
        # 300m 격자 ID 컬럼명 통일
        monthly_df = monthly_df.rename(
            columns={"격자 아이디": "300m 격자 ID"}
        )

        # 기준년월 컬럼명 통일 
        id_columns = [
            "기준년월",
            "300m 격자 ID",
            "데이터기준일자",
        ]

        time_frames = []
        # 24시간 동안의 시간대별 유동인구와 비율을 행 구조로 변환
        for hour in range(24):
            next_hour = (hour + 1) % 24 #(다음 시간대 계산, 23시 다음은 0시)
            time_slot = f"{hour:02d}시~{next_hour:02d}시"
            population_column = f"{time_slot} 유동인구"
            ratio_column = f"{time_slot} 유동인구 비율"

            time_df = monthly_df[id_columns].copy()
            time_df["time_slot"] = time_slot # 현재 처리 중인 시간대 추가
            # 현재 시간대의 유동인구와 비율 컬럼을 새로운 컬럼으로 추가
            time_df["flow_population"] = monthly_df[population_column]
            time_df["flow_ratio"] = monthly_df[ratio_column]
            # 시간대별 데이터프레임을 리스트에 추가
            time_frames.append(time_df)

        # 12개월 동안의 시간대별 데이터프레임을 하나로 합치기
        monthly_long_df = pd.concat(
            time_frames,
            ignore_index=True,
        )
        monthly_frames.append(monthly_long_df)

        print(
            Path(file_name).name,
            "변환 완료:",
            monthly_long_df.shape,
        )

grid_df = pd.concat(
    monthly_frames,
    ignore_index=True,
)

제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140131.csv 변환 완료: (210240, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140228.csv 변환 완료: (218424, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140331.csv 변환 완료: (222792, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140430.csv 변환 완료: (225336, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140531.csv 변환 완료: (225312, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140630.csv 변환 완료: (242640, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140731.csv 변환 완료: (244488, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140831.csv 변환 완료: (248256, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20140930.csv 변환 완료: (251952, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141031.csv 변환 완료: (255552, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141130.csv 변환 완료: (255024, 6)
제주특별자치도_시간대별_관광객_300그리드단위_유동인구_비율포함_20141231.csv 변환 완료: (260640, 6)


In [ ]:
# 컬럼명과 자료형 정리
grid_df = grid_df.rename(
    columns={
        "기준년월": "reference_month",
        "300m 격자 ID": "grid_id",
        "데이터기준일자": "data_date",
    }
)

# 원본의 잘못된 연월 표기 수정 : 2014--04, 2014--05를 정상적인 연월 형식으로 통일
grid_df["reference_month"] = (
    grid_df["reference_month"]
    .str.replace("--", "-", regex=False)
)
# 기준년월을 날짜 자료형으로 바꿈
grid_df["reference_month"] = pd.to_datetime(
    grid_df["reference_month"],
    format="%Y-%m",
    errors="coerce",
)

# 데이터기준일자를 날짜 자료형으로 바꿈
grid_df["data_date"] = pd.to_datetime(
    grid_df["data_date"],
    errors="coerce",
)

# 300m 격자 ID를 문자열 자료형으로 바꿈 (격자 ID는 숫자처럼 보이지만 실제로는 문자열로 취급해야 함)
grid_df["grid_id"] = grid_df["grid_id"].astype("string")

# 유동인구와 비율을 숫자형으로 변환
for column in ["flow_population", "flow_ratio"]:
    grid_df[column] = pd.to_numeric(
        grid_df[column],
        errors="coerce",
    )

# 데이터 프로파일링 수행
clean_df = etl.transform(grid_df)
# 필요한 컬럼만 선택,순서 고정
clean_df = clean_df[
    [
        "reference_month",
        "data_date",
        "grid_id",
        "time_slot",
        "flow_population",
        "flow_ratio",
    ]
]

# 데이터 정렬: 기준년월, 격자ID, 시간대 순으로 정렬하고 인덱스 재설정
clean_df = clean_df.sort_values(
    ["reference_month", "grid_id", "time_slot"]
).reset_index(drop=True)

print("최종 크기:", clean_df.shape)
print("최종 컬럼:", clean_df.columns.tolist())
print("자료형:")
print(clean_df.dtypes)
display(clean_df.head())

최종 크기: (2860656, 6)
최종 컬럼: ['reference_month', 'data_date', 'grid_id', 'time_slot', 'flow_population', 'flow_ratio']
자료형:
reference_month    datetime64[ns]
data_date          datetime64[ns]
grid_id            string[python]
time_slot                  object
flow_population           float64
flow_ratio                float64
dtype: object


,reference_month,data_date,grid_id,time_slot,flow_population,flow_ratio
0,2014-01-01,2014-01-31,1000007915,00시~01시,2.920323,0.000161
1,2014-01-01,2014-01-31,1000007915,01시~02시,1.976452,0.000145
2,2014-01-01,2014-01-31,1000007915,02시~03시,1.969677,0.000166
3,2014-01-01,2014-01-31,1000007915,03시~04시,2.232581,0.000216
4,2014-01-01,2014-01-31,1000007915,04시~05시,2.480323,0.000266


In [ ]:
# 전체 데이터의 기본 품질 정보

#각 컬럼의 결측값 개수를 딕셔너리로
full_missing_count = {
    column: int(count)
    for column, count in clean_df.isna().sum().items()
}
# 각 컬럼의 결측값 비율을 딕셔너리로
full_missing_rate = {
    column: float(rate)
    for column, rate in clean_df.isna().mean().items()
}
# 완전히 동일한 중복 행을 계산
full_duplicate_count = int(
    clean_df.duplicated().sum()
)

print("컬럼별 결측값:", full_missing_count)
print("중복 행 수:", full_duplicate_count)
print("기준월 범위:", clean_df["reference_month"].min(), "~", clean_df["reference_month"].max())

컬럼별 결측값: {'reference_month': 0, 'data_date': 0, 'grid_id': 0, 'time_slot': 0, 'flow_population': 0, 'flow_ratio': 0}
중복 행 수: 0
기준월 범위: 2014-01-01 00:00:00 ~ 2014-12-01 00:00:00


In [6]:
# 전체 데이터 저장
saved_parquet_path = etl.save_to_parquet(
    clean_df,
    parquet_output_path,
)

print("Parquet 저장 완료:", saved_parquet_path)

Parquet 저장 완료: outputs\jeju_grid_300m.parquet


In [ ]:
# 전체 데이터 286만행
# ydata-profiling은 커널 과부하를 피하기 위해 최대 100,000행 표본에 적용
profile_sample_size = min(100_000, len(clean_df))
# 전체 데이터에서 10만행을 무작위로 뽑음
profile_df = clean_df.sample(
    n=profile_sample_size,
    random_state=42,
)

# 프로파일링
stats = profiler.profile_data(profile_df)

# 행 수·결측값·중복 수는 전체 데이터에서 계산한 정확한 값으로 교체

#표본 크기가 아니라 전체 데이터의 행,열 정보를 넣음
stats["row_count"] = int(len(clean_df))
stats["column_count"] = int(clean_df.shape[1])
#결측값과 중복 수도 전체 데이터에서 계산한 값을 넣음
stats["column_names"] = clean_df.columns.tolist()
stats["missing_count"] = full_missing_count
stats["missing_rate"] = full_missing_rate
#전체 셀 중 결측 셀이 차지하는 비율
stats["overall_missing_rate"] = float(
    clean_df.isna().sum().sum()
    / clean_df.size
)
stats["duplicate_count"] = full_duplicate_count
stats["profiling_sample_size"] = profile_sample_size
stats["type_inference_scope"] = "100,000-row sample"

# 태그 생성
tags = profiler.generate_tags(stats)

print("프로파일 결과:")
print(json.dumps(stats, ensure_ascii=False, indent=2))
print("생성된 태그:", tags)

100%|██████████| 6/6 [00:00<00:00, 23.52it/s]


프로파일 결과:
{
  "row_count": 2860656,
  "column_count": 6,
  "column_names": [
    "reference_month",
    "data_date",
    "grid_id",
    "time_slot",
    "flow_population",
    "flow_ratio"
  ],
  "data_types": {
    "reference_month": "DateTime",
    "data_date": "DateTime",
    "grid_id": "Text",
    "time_slot": "Text",
    "flow_population": "Numeric",
    "flow_ratio": "Numeric"
  },
  "missing_count": {
    "reference_month": 0,
    "data_date": 0,
    "grid_id": 0,
    "time_slot": 0,
    "flow_population": 0,
    "flow_ratio": 0
  },
  "missing_rate": {
    "reference_month": 0.0,
    "data_date": 0.0,
    "grid_id": 0.0,
    "time_slot": 0.0,
    "flow_population": 0.0,
    "flow_ratio": 0.0
  },
  "overall_missing_rate": 0.0,
  "duplicate_count": 0,
  "numeric_columns": [
    "flow_population",
    "flow_ratio"
  ],
  "numeric_column_count": 2,
  "datetime_columns": [
    "reference_month",
    "data_date"
  ],
  "has_datetime": true,
  "categorical_columns": [
    "grid_id",
 

In [ ]:
# 메타데이터 JSON 저장
# 데이터셋의 기본 정보
dataset_info = {
    "id": "JEJU_GRID_300M_2014_01",
    "name": "제주 300m 그리드 시간대별 관광객 유동인구",
    "source_file": input_path.name,
    "selected_csv_count": len(base_csv_files),
}

# JSON 파일로 저장
saved_metadata_path = profiler.export_metadata(
    dataset_info=dataset_info,
    profile_stats=stats,
    tags=tags,
    out_meta_path=metadata_output_path,
)

print("메타데이터 저장 완료:", saved_metadata_path)

메타데이터 저장 완료: outputs\jeju_grid_300m_metadata.json


In [ ]:
# 저장 결과 재로드 및 검증
parquet_check = pd.read_parquet(saved_parquet_path)

with saved_metadata_path.open("r", encoding="utf-8") as file:
    metadata_check = json.load(file)

# 검증
assert parquet_check.shape == clean_df.shape
assert parquet_check.columns.tolist() == clean_df.columns.tolist()
assert len(parquet_check) == 2_860_656
assert parquet_check["reference_month"].notna().all()
assert metadata_check["dataset_info"] == dataset_info
assert metadata_check["profile_stats"] == stats
assert metadata_check["tags"] == tags

print("Parquet 재로드 결과:", parquet_check.shape)
print("메타데이터 태그:", metadata_check["tags"])
print("300m 그리드 파이프라인 검증 통과")

Parquet 재로드 결과: (2860656, 6)
메타데이터 태그: ['time-series', 'multivariate']
300m 그리드 파이프라인 검증 통과
